In [1]:
import kagglehub
import os

# Set your custom download path
custom_path = r"DIOR"  # Change to your desired path

# Create directory if it doesn't exist
os.makedirs(custom_path, exist_ok=True)

# Download dataset to default location first
path = kagglehub.dataset_download("shuaitt/diordata")

print("Dataset downloaded to:", path)

# Move files to custom location (optional)
import shutil
for item in os.listdir(path):
    src = os.path.join(path, item)
    dst = os.path.join(custom_path, item)
    shutil.move(src, dst)

print(f"Files moved to: {custom_path}")

/home/gpu_03/class-agnostic/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Dataset downloaded to: /home/gpu_03/.cache/kagglehub/datasets/shuaitt/diordata/versions/1
Files moved to: DIOR


In [5]:
import os
import shutil
from pathlib import Path
from collections import defaultdict

# Define splits and their corresponding text files
SPLITS = {
    'train': 'train.txt',
    'val': 'val.txt',
    'trainval': 'trainval.txt',
    'test': 'test.txt'
}

def read_image_names_from_txt(txt_path):
    """
    Read image names (without extension) from a text file.
    """
    if not os.path.exists(txt_path):
        print(f"Warning: {txt_path} not found")
        return set()
    
    with open(txt_path, 'r') as f:
        # Read lines and strip whitespace, remove empty lines
        names = set(line.strip() for line in f if line.strip())
    
    return names

def create_split_folders(output_dir, split_name):
    """
    Create folder structure for a specific split.
    """
    split_dir = os.path.join(output_dir, split_name)
    images_split_dir = os.path.join(split_dir, "images")
    labels_split_dir = os.path.join(split_dir, "labels")
    
    os.makedirs(images_split_dir, exist_ok=True)
    os.makedirs(labels_split_dir, exist_ok=True)
    
    return images_split_dir, labels_split_dir

def copy_files_for_split(image_names, source_images_dir, source_labels_dir, 
                         dest_images_dir, dest_labels_dir, split_name):
    """
    Copy images and labels for a specific split.
    """
    copied_count = 0
    missing_labels = []
    missing_images = []
    
    for name in image_names:
        # Find image file (supports different extensions)
        img_file = None
        for ext in ['.jpg', '.jpeg', '.png', '.bmp', '.tiff']:
            potential_img = os.path.join(source_images_dir, f"{name}{ext}")
            if os.path.exists(potential_img):
                img_file = f"{name}{ext}"
                break
        
        if img_file is None:
            missing_images.append(name)
            continue
        
        # Copy image
        src_img = os.path.join(source_images_dir, img_file)
        dst_img = os.path.join(dest_images_dir, img_file)
        shutil.copy2(src_img, dst_img)
        
        # Copy corresponding label
        label_file = f"{name}.txt"
        src_label = os.path.join(source_labels_dir, label_file)
        dst_label = os.path.join(dest_labels_dir, label_file)
        
        if os.path.exists(src_label):
            shutil.copy2(src_label, dst_label)
        else:
            missing_labels.append(name)
        
        copied_count += 1
    
    # Report issues
    if missing_images:
        print(f"  ⚠️  {split_name}: Missing {len(missing_images)} images")
        if len(missing_images) <= 5:
            print(f"     Missing: {missing_images}")
    
    if missing_labels:
        print(f"  ⚠️  {split_name}: Missing {len(missing_labels)} labels")
        if len(missing_labels) <= 5:
            print(f"     Missing: {missing_labels}")
    
    return copied_count

def validate_splits(split_data):
    """
    Validate that splits don't have overlapping images.
    """
    all_splits = {}
    overlaps = defaultdict(list)
    
    for split_name, image_names in split_data.items():
        all_splits[split_name] = image_names
    
    # Check for overlaps between different splits
    split_names = list(all_splits.keys())
    for i in range(len(split_names)):
        for j in range(i+1, len(split_names)):
            split1 = split_names[i]
            split2 = split_names[j]
            overlap = all_splits[split1].intersection(all_splits[split2])
            if overlap:
                overlaps[f"{split1} & {split2}"] = overlap
    
    return overlaps

def print_split_statistics(split_data, copied_counts):
    """
    Print detailed statistics about each split.
    """
    print("\n" + "="*70)
    print("SPLIT STATISTICS")
    print("="*70)
    
    total_images = 0
    total_copied = 0
    
    for split_name in SPLITS.keys():
        image_names = split_data.get(split_name, set())
        copied = copied_counts.get(split_name, 0)
        
        total_images += len(image_names)
        total_copied += copied
        
        print(f"\n{split_name.upper()}:")
        print(f"  - Images in text file: {len(image_names)}")
        print(f"  - Successfully copied: {copied}")
        print(f"  - Success rate: {copied/len(image_names)*100:.1f}%" if image_names else "  - Success rate: N/A")
    
    print(f"\n{'='*70}")
    print(f"TOTAL:")
    print(f"  - Total images across all splits: {total_images}")
    print(f"  - Total successfully copied: {total_copied}")
    
    # Check for duplicates
    print(f"\n{'='*70}")
    print("DUPLICATE CHECK")
    print("="*70)
    
    overlaps = validate_splits(split_data)
    if overlaps:
        print("⚠️  Warning: Overlapping images found between splits:")
        for split_pair, overlap_set in overlaps.items():
            print(f"  - {split_pair}: {len(overlap_set)} overlapping images")
            if len(overlap_set) <= 5:
                print(f"    Images: {list(overlap_set)}")
    else:
        print("✓ No overlapping images between splits")

def verify_output_structure(output_dir):
    """
    Verify the output directory structure.
    """
    print("\n" + "="*70)
    print("OUTPUT STRUCTURE")
    print("="*70)
    print(f"Root directory: {output_dir}")
    
    for split_name in SPLITS.keys():
        split_path = os.path.join(output_dir, split_name)
        images_path = os.path.join(split_path, "images")
        labels_path = os.path.join(split_path, "labels")
        
        if os.path.exists(split_path):
            num_images = len([f for f in os.listdir(images_path) 
                             if f.endswith(('.jpg', '.jpeg', '.png'))]) if os.path.exists(images_path) else 0
            num_labels = len([f for f in os.listdir(labels_path) 
                             if f.endswith('.txt')]) if os.path.exists(labels_path) else 0
            
            print(f"\n{split_name}/")
            print(f"  ├── images/  ({num_images} files)")
            print(f"  └── labels/  ({num_labels} files)")
        else:
            print(f"\n{split_name}/  (not created - no files)")

def main():
    """
    Main function to split dataset based on ImageSets files.
    """
    print("="*70)
    print("DATASET SPLITTER TOOL")
    print("="*70)
    print(f"\nBase directory: {BASE_DIR}")
    print(f"Images directory: {IMAGES_DIR}")
    print(f"Labels directory: {LABELS_DIR}")
    print(f"ImageSets directory: {IMAGESETS_DIR}")
    print(f"Output directory: {OUTPUT_DIR}")
    
    # Check if directories exist
    if not os.path.exists(IMAGES_DIR):
        print(f"\n❌ Error: Images directory not found: {IMAGES_DIR}")
        return
    
    if not os.path.exists(LABELS_DIR):
        print(f"\n⚠️  Warning: Labels directory not found: {LABELS_DIR}")
    
    if not os.path.exists(IMAGESETS_DIR):
        print(f"\n❌ Error: ImageSets directory not found: {IMAGESETS_DIR}")
        return
    
    # Read all split files
    print("\n" + "="*70)
    print("READING SPLIT FILES")
    print("="*70)
    
    split_data = {}
    for split_name, txt_file in SPLITS.items():
        txt_path = os.path.join(IMAGESETS_DIR, txt_file)
        image_names = read_image_names_from_txt(txt_path)
        split_data[split_name] = image_names
        print(f"\n{split_name.upper()}: {len(image_names)} images from {txt_file}")
    
    # Create output directory and copy files
    print("\n" + "="*70)
    print("COPYING FILES")
    print("="*70)
    
    copied_counts = {}
    
    for split_name, image_names in split_data.items():
        if not image_names:
            print(f"\n⚠️  Skipping {split_name} (no images found)")
            copied_counts[split_name] = 0
            continue
        
        print(f"\nProcessing {split_name.upper()} split...")
        
        # Create folders for this split
        dest_images_dir, dest_labels_dir = create_split_folders(OUTPUT_DIR, split_name)
        
        # Copy files
        copied = copy_files_for_split(
            image_names, IMAGES_DIR, LABELS_DIR,
            dest_images_dir, dest_labels_dir, split_name
        )
        
        copied_counts[split_name] = copied
        print(f"  ✓ Copied {copied}/{len(image_names)} images with labels")
    
    # Print statistics
    print_split_statistics(split_data, copied_counts)
    
    # Verify output structure
    verify_output_structure(OUTPUT_DIR)
    
    print("\n" + "="*70)
    print("✅ PROCESS COMPLETED SUCCESSFULLY!")
    print("="*70)

if __name__ == "__main__":
    # Update this path to your actual folder
    BASE_DIR = "DIOR"  # <-- CHANGE THIS TO YOUR PATH
    
    # Then update the dependent paths
    IMAGES_DIR = os.path.join(BASE_DIR, "images")
    LABELS_DIR = os.path.join(BASE_DIR, "labels")
    IMAGESETS_DIR = os.path.join(BASE_DIR, "ImageSets")
    OUTPUT_DIR = os.path.join(BASE_DIR, "dior_split")
    
    main()

DATASET SPLITTER TOOL

Base directory: DIOR
Images directory: DIOR/images
Labels directory: DIOR/labels
ImageSets directory: DIOR/ImageSets
Output directory: DIOR/dior_split

READING SPLIT FILES

TRAIN: 19004 images from train.txt

VAL: 2112 images from val.txt

TRAINVAL: 21116 images from trainval.txt

TEST: 2347 images from test.txt

COPYING FILES

Processing TRAIN split...
  ✓ Copied 19004/19004 images with labels

Processing VAL split...
  ✓ Copied 2112/2112 images with labels

Processing TRAINVAL split...
  ✓ Copied 21116/21116 images with labels

Processing TEST split...
  ✓ Copied 2347/2347 images with labels

SPLIT STATISTICS

TRAIN:
  - Images in text file: 19004
  - Successfully copied: 19004
  - Success rate: 100.0%

VAL:
  - Images in text file: 2112
  - Successfully copied: 2112
  - Success rate: 100.0%

TRAINVAL:
  - Images in text file: 21116
  - Successfully copied: 21116
  - Success rate: 100.0%

TEST:
  - Images in text file: 2347
  - Successfully copied: 2347
  - Suc

In [1]:
import os
import shutil
from pathlib import Path
from collections import defaultdict

def get_classes_from_label(label_path):
    """
    Extract unique class IDs from a label file.
    """
    classes = set()
    if not os.path.exists(label_path):
        return classes
    
    with open(label_path, 'r') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            parts = line.split()
            if len(parts) >= 1:
                try:
                    class_id = int(parts[0])
                    classes.add(class_id)
                except ValueError:
                    continue
    return classes

def filter_split_keep_structure(split_name, split_path, choice_classes, output_base_dir):
    """
    Filter a single split while maintaining structure.
    Returns selected images and class statistics.
    """
    images_dir = os.path.join(split_path, "images")
    labels_dir = os.path.join(split_path, "labels")
    
    if not os.path.exists(images_dir):
        print(f"  ⚠️  {split_name}: Images directory not found")
        return 0, 0, 0, defaultdict(int)
    
    # Create output directories for this split
    output_split_dir = os.path.join(output_base_dir, split_name)
    output_images_dir = os.path.join(output_split_dir, "images")
    output_labels_dir = os.path.join(output_split_dir, "labels")
    os.makedirs(output_images_dir, exist_ok=True)
    os.makedirs(output_labels_dir, exist_ok=True)
    
    # Get all image files
    image_extensions = {'.jpg', '.jpeg', '.png', '.bmp', '.tiff'}
    image_files = [f for f in os.listdir(images_dir) 
                   if Path(f).suffix.lower() in image_extensions]
    
    selected_images = []
    class_counts = defaultdict(int)
    filtered_out_count = 0
    missing_label_count = 0
    
    for img_file in image_files:
        base_name = Path(img_file).stem
        label_file = f"{base_name}.txt"
        label_path = os.path.join(labels_dir, label_file)
        
        if not os.path.exists(label_path):
            missing_label_count += 1
            filtered_out_count += 1
            continue
        
        # Get classes in this image
        classes_in_image = get_classes_from_label(label_path)
        
        # Check if image contains any choice class
        matching_classes = classes_in_image.intersection(choice_classes)
        
        if matching_classes:
            # Copy image and label to output
            shutil.copy2(os.path.join(images_dir, img_file), 
                        os.path.join(output_images_dir, img_file))
            shutil.copy2(label_path, 
                        os.path.join(output_labels_dir, label_file))
            
            selected_images.append(img_file)
            
            # Count classes
            for cls in matching_classes:
                class_counts[cls] += 1
        else:
            filtered_out_count += 1
    
    return len(selected_images), filtered_out_count, missing_label_count, class_counts

def print_split_statistics(output_dir, choice_classes):
    """
    Print detailed statistics for each split.
    """
    print("\n" + "="*70)
    print("FILTERED DATASET STATISTICS")
    print("="*70)
    
    split_dirs = [d for d in os.listdir(output_dir) 
                  if os.path.isdir(os.path.join(output_dir, d))]
    
    total_stats = {}
    overall_class_counts = defaultdict(int)
    
    for split_name in sorted(split_dirs):
        images_dir = os.path.join(output_dir, split_name, "images")
        labels_dir = os.path.join(output_dir, split_name, "labels")
        
        if not os.path.exists(images_dir):
            continue
        
        num_images = len([f for f in os.listdir(images_dir) 
                         if f.endswith(('.jpg', '.jpeg', '.png'))])
        num_labels = len([f for f in os.listdir(labels_dir) 
                         if f.endswith('.txt')])
        
        # Count class distribution in this split
        class_counts = defaultdict(int)
        for label_file in os.listdir(labels_dir):
            if label_file.endswith('.txt'):
                label_path = os.path.join(labels_dir, label_file)
                classes = get_classes_from_label(label_path)
                for cls in classes.intersection(choice_classes):
                    class_counts[cls] += 1
                    overall_class_counts[cls] += 1
        
        total_stats[split_name] = {
            'images': num_images,
            'labels': num_labels,
            'class_counts': class_counts
        }
    
    # Print per-split statistics
    print("\nPer-split summary:")
    print("-" * 70)
    for split_name, stats in total_stats.items():
        print(f"\n{split_name.upper()}:")
        print(f"  Total images: {stats['images']}")
        if stats['class_counts']:
            print(f"  Class distribution:")
            for cls in sorted(choice_classes):
                count = stats['class_counts'].get(cls, 0)
                if count > 0:
                    print(f"    Class {cls:2d}: {count:3d} instances")
        else:
            print(f"  No images with choice classes found")
    
    # Print overall statistics
    print("\n" + "="*70)
    print("OVERALL CLASS DISTRIBUTION")
    print("="*70)
    for cls in sorted(choice_classes):
        count = overall_class_counts.get(cls, 0)
        if count > 0:
            bar_length = min(50, count // 5)
            bar = "█" * bar_length
            print(f"Class {cls:2d}: {count:3d} instances {bar}")
        else:
            print(f"Class {cls:2d}: {count:3d} instances (no images found)")
    
    return total_stats

def verify_split_integrity(output_dir, choice_classes):
    """
    Verify that all images in the filtered dataset contain at least one choice class.
    """
    print("\n" + "="*70)
    print("VERIFICATION")
    print("="*70)
    
    split_dirs = [d for d in os.listdir(output_dir) 
                  if os.path.isdir(os.path.join(output_dir, d))]
    
    total_verified = 0
    errors = []
    
    for split_name in split_dirs:
        labels_dir = os.path.join(output_dir, split_name, "labels")
        if not os.path.exists(labels_dir):
            continue
        
        for label_file in os.listdir(labels_dir):
            if not label_file.endswith('.txt'):
                continue
            
            label_path = os.path.join(labels_dir, label_file)
            classes = get_classes_from_label(label_path)
            matching = classes.intersection(choice_classes)
            
            if not matching:
                errors.append(f"{split_name}/{label_file}")
            else:
                total_verified += 1
    
    if errors:
        print(f"⚠️  Found {len(errors)} images without choice classes:")
        for err in errors[:10]:
            print(f"   - {err}")
        return False
    else:
        print(f"✓ Verified {total_verified} images - all contain at least one choice class!")
        return True

def create_dataset_summary(output_dir, choice_classes):
    """
    Create a summary text file with dataset information.
    """
    summary_path = os.path.join(output_dir, "dataset_summary.txt")
    
    with open(summary_path, 'w') as f:
        f.write("="*70 + "\n")
        f.write("FILTERED DATASET SUMMARY\n")
        f.write("="*70 + "\n\n")
        
        f.write(f"Choice classes: {choice_classes}\n\n")
        
        # Get split statistics
        split_dirs = [d for d in os.listdir(output_dir) 
                      if os.path.isdir(os.path.join(output_dir, d))]
        
        total_images = 0
        
        for split_name in sorted(split_dirs):
            images_dir = os.path.join(output_dir, split_name, "images")
            labels_dir = os.path.join(output_dir, split_name, "labels")
            
            if not os.path.exists(images_dir):
                continue
            
            num_images = len([f for f in os.listdir(images_dir) 
                             if f.endswith(('.jpg', '.jpeg', '.png'))])
            total_images += num_images
            
            f.write(f"\n{split_name.upper()} split:\n")
            f.write(f"  Number of images: {num_images}\n")
            
            # Count class distribution
            class_counts = defaultdict(int)
            for label_file in os.listdir(labels_dir):
                if label_file.endswith('.txt'):
                    label_path = os.path.join(labels_dir, label_file)
                    classes = get_classes_from_label(label_path)
                    for cls in classes.intersection(choice_classes):
                        class_counts[cls] += 1
            
            if class_counts:
                f.write(f"  Class distribution:\n")
                for cls in sorted(choice_classes):
                    count = class_counts.get(cls, 0)
                    if count > 0:
                        f.write(f"    Class {cls}: {count} instances\n")
            else:
                f.write(f"  No images with choice classes found\n")
        
        f.write("\n" + "="*70 + "\n")
        f.write(f"Total images across all splits: {total_images}\n")
    
    print(f"\n✓ Dataset summary saved to: {summary_path}")

def main():
    """
    Main function to filter dataset while maintaining train/val/test split.
    """
    print("="*70)
    print("FILTERING DATASET WITH SPLIT PRESERVATION")
    print("="*70)
    print(f"\nChoice classes: {CHOICE_CLASSES}")
    print(f"Source dataset: {SPLIT_DATASET_DIR}")
    print(f"Output directory: {OUTPUT_DIR}")
    
    # Check if source directory exists
    if not os.path.exists(SPLIT_DATASET_DIR):
        print(f"\n❌ Error: Split dataset directory not found: {SPLIT_DATASET_DIR}")
        print("Please update the SPLIT_DATASET_DIR variable with the correct path.")
        return
    
    # Get all split folders from source
    split_folders = [d for d in os.listdir(SPLIT_DATASET_DIR) 
                    if os.path.isdir(os.path.join(SPLIT_DATASET_DIR, d))]
    
    if not split_folders:
        print(f"\n❌ Error: No split folders found in {SPLIT_DATASET_DIR}")
        return
    
    print(f"\nFound splits: {split_folders}")
    
    # Filter each split
    print("\n" + "="*70)
    print("FILTERING EACH SPLIT")
    print("="*70)
    
    total_stats = {}
    all_class_counts = defaultdict(int)
    
    for split_name in sorted(split_folders):
        split_path = os.path.join(SPLIT_DATASET_DIR, split_name)
        print(f"\nProcessing {split_name.upper()}...")
        
        selected, filtered_out, missing_labels, class_counts = filter_split_keep_structure(
            split_name, split_path, CHOICE_CLASSES, OUTPUT_DIR
        )
        
        total_stats[split_name] = {
            'selected': selected,
            'filtered_out': filtered_out,
            'missing_labels': missing_labels
        }
        
        for cls, count in class_counts.items():
            all_class_counts[cls] += count
        
        print(f"  ✓ Kept: {selected} images (contain choice classes)")
        print(f"  ✗ Removed: {filtered_out} images (no choice classes)")
        if missing_labels > 0:
            print(f"  ⚠️  Missing labels: {missing_labels} images")
    
    # Check if any images were selected
    if not any(stats['selected'] > 0 for stats in total_stats.values()):
        print("\n❌ No images found with the specified classes!")
        return
    
    # Print statistics
    split_stats = print_split_statistics(OUTPUT_DIR, CHOICE_CLASSES)
    
    # Verify integrity
    verify_passed = verify_split_integrity(OUTPUT_DIR, CHOICE_CLASSES)
    
    # Create summary file
    create_dataset_summary(OUTPUT_DIR, CHOICE_CLASSES)
    
    # Final summary
    print("\n" + "="*70)
    print("PROCESS COMPLETED")
    print("="*70)
    
    total_images = sum(stats['selected'] for stats in total_stats.values())
    print(f"\n✅ Filtered dataset created successfully!")
    print(f"📁 Location: {OUTPUT_DIR}")
    print(f"📊 Total images: {total_images}")
    print(f"🎯 Classes preserved: {CHOICE_CLASSES}")
    
    print("\nSplit structure maintained:")
    for split_name in sorted(split_folders):
        if total_stats.get(split_name, {}).get('selected', 0) > 0:
            print(f"  ✓ {split_name}/")
            print(f"    ├── images/  ({total_stats[split_name]['selected']} files)")
            print(f"    └── labels/  ({total_stats[split_name]['selected']} files)")
        elif total_stats.get(split_name, {}).get('selected', 0) == 0:
            print(f"  ✓ {split_name}/ (empty - no matching images)")

# Update these paths before running
SPLIT_DATASET_DIR = r"DIOR/dior_split"  # <-- CHANGE THIS
OUTPUT_DIR = r"DIOR/dior_filtered_classes"       # <-- CHANGE THIS
CHOICE_CLASSES = [0,4, 7, 8]

main()

FILTERING DATASET WITH SPLIT PRESERVATION

Choice classes: [0, 4, 7, 8]
Source dataset: DIOR/dior_split
Output directory: DIOR/dior_filtered_classes

Found splits: ['trainval', 'val', 'test', 'train']

FILTERING EACH SPLIT

Processing TEST...
  ✓ Kept: 679 images (contain choice classes)
  ✗ Removed: 1668 images (no choice classes)

Processing TRAIN...


  ✓ Kept: 5826 images (contain choice classes)
  ✗ Removed: 13178 images (no choice classes)

Processing TRAINVAL...
  ✓ Kept: 6472 images (contain choice classes)
  ✗ Removed: 14644 images (no choice classes)

Processing VAL...
  ✓ Kept: 646 images (contain choice classes)
  ✗ Removed: 1466 images (no choice classes)

FILTERED DATASET STATISTICS

Per-split summary:
----------------------------------------------------------------------

TEST:
  Total images: 679
  Class distribution:
    Class  0: 148 instances
    Class  4: 155 instances
    Class  7: 262 instances
    Class  8: 133 instances

TRAIN:
  Total images: 5826
  Class distribution:
    Class  0: 1377 instances
    Class  4: 1318 instances
    Class  7: 2178 instances
    Class  8: 1135 instances

TRAINVAL:
  Total images: 6472
  Class distribution:
    Class  0: 1520 instances
    Class  4: 1461 instances
    Class  7: 2440 instances
    Class  8: 1254 instances

VAL:
  Total images: 646
  Class distribution:
    Class  0: 

In [3]:
import os
import shutil
import random
from pathlib import Path
from collections import defaultdict

def get_classes_from_label(label_path):
    """
    Extract all class IDs from a label file.
    """
    classes = set()
    if not os.path.exists(label_path):
        return classes
    
    with open(label_path, 'r') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            parts = line.split()
            if len(parts) >= 1:
                try:
                    class_id = int(parts[0])
                    classes.add(class_id)
                except ValueError:
                    continue
    return classes

def collect_images_by_class(image_dir, label_dir, classes):
    """
    Collect all images and group them by the classes they contain.
    """
    # Get all image files
    image_extensions = {'.jpg', '.jpeg', '.png', '.bmp', '.tiff'}
    image_files = [f for f in os.listdir(image_dir) 
                   if Path(f).suffix.lower() in image_extensions]
    
    # Dictionary to store images for each class
    class_to_images = defaultdict(list)
    
    # Also store which classes each image contains
    image_classes = {}
    
    print("Scanning images and labels...")
    for img_file in image_files:
        base_name = Path(img_file).stem
        label_file = f"{base_name}.txt"
        label_path = os.path.join(label_dir, label_file)
        
        if not os.path.exists(label_path):
            print(f"  Warning: No label found for {img_file}")
            continue
        
        # Get classes in this image
        classes_in_image = get_classes_from_label(label_path)
        
        if classes_in_image:
            image_classes[img_file] = classes_in_image
            
            # Add this image to each class it belongs to
            for cls in classes_in_image:
                if cls in classes:
                    class_to_images[cls].append(img_file)
    
    return class_to_images, image_classes

def select_images_per_class(class_to_images, images_per_class=10):
    """
    Select specified number of images for each class.
    """
    selected_images = set()
    class_selection = {}
    
    print("\n" + "="*60)
    print("SELECTING IMAGES FOR EACH CLASS")
    print("="*60)
    
    for cls in sorted(class_to_images.keys()):
        available_images = class_to_images[cls]
        available_count = len(available_images)
        
        # Determine how many to select
        select_count = min(images_per_class, available_count)
        
        if select_count < images_per_class:
            print(f"\nClass {cls}: Only {available_count} images available (need {images_per_class})")
            print(f"  Selecting all {select_count} images")
        else:
            print(f"\nClass {cls}: {available_count} images available")
            print(f"  Selecting {select_count} images")
        
        # Randomly select images
        random.seed(42)  # For reproducibility, remove for random each time
        selected = random.sample(available_images, select_count)
        
        class_selection[cls] = {
            'selected': selected,
            'count': select_count,
            'available': available_count
        }
        
        # Add to overall selected set
        selected_images.update(selected)
    
    return selected_images, class_selection

def copy_selected_images(selected_images, image_dir, label_dir, output_dir, image_classes):
    """
    Copy selected images and their labels to output directory.
    """
    # Create output directories
    output_images_dir = os.path.join(output_dir, "images")
    output_labels_dir = os.path.join(output_dir, "labels")
    os.makedirs(output_images_dir, exist_ok=True)
    os.makedirs(output_labels_dir, exist_ok=True)
    
    copied_count = 0
    copied_classes = defaultdict(int)
    
    print("\n" + "="*60)
    print("COPYING FILES")
    print("="*60)
    
    for img_file in selected_images:
        base_name = Path(img_file).stem
        
        # Copy image
        src_img = os.path.join(image_dir, img_file)
        dst_img = os.path.join(output_images_dir, img_file)
        shutil.copy2(src_img, dst_img)
        
        # Copy label
        label_file = f"{base_name}.txt"
        src_label = os.path.join(label_dir, label_file)
        dst_label = os.path.join(output_labels_dir, label_file)
        
        if os.path.exists(src_label):
            shutil.copy2(src_label, dst_label)
        
        # Count classes for statistics
        for cls in image_classes[img_file]:
            copied_classes[cls] += 1
        
        copied_count += 1
    
    return copied_count, copied_classes

def print_statistics(class_selection, copied_classes, output_dir):
    """
    Print detailed statistics about the selection.
    """
    print("\n" + "="*60)
    print("SELECTION STATISTICS")
    print("="*60)
    
    print("\nPer-class selection summary:")
    print("-" * 50)
    
    total_selected = 0
    total_available = 0
    
    for cls in sorted(class_selection.keys()):
        info = class_selection[cls]
        selected_count = info['count']
        available_count = info['available']
        
        total_selected += selected_count
        total_available += available_count
        
        status = "✓" if selected_count >= IMAGES_PER_CLASS else "⚠️"
        print(f"Class {cls:2d}: {selected_count:2d}/{IMAGES_PER_CLASS} selected "
              f"(Available: {available_count}) {status}")
    
    print(f"\nTotal unique images selected: {len(class_selection)}")
    print(f"Total image instances (counting multiple classes per image): {total_selected}")
    
    # Class distribution in copied images (accounting for multi-class images)
    print("\n" + "="*60)
    print("ACTUAL CLASS DISTRIBUTION IN COPIED IMAGES")
    print("="*60)
    print("(Note: One image may contain multiple classes)")
    print("-" * 50)
    
    for cls in sorted(class_selection.keys()):
        count = copied_classes.get(cls, 0)
        bar_length = min(40, count // 2)
        bar = "█" * bar_length
        print(f"Class {cls:2d}: {count:3d} instances {bar}")
    
    # Output structure
    print("\n" + "="*60)
    print("OUTPUT STRUCTURE")
    print("="*60)
    print(f"\nOutput directory: {output_dir}")
    print(f"  ├── images/  ({len(class_selection)} image files)")
    print(f"  └── labels/  ({len(class_selection)} label files)")

def verify_selection(output_dir, expected_classes):
    """
    Verify that selected images contain the expected classes.
    """
    print("\n" + "="*60)
    print("VERIFICATION")
    print("="*60)
    
    labels_dir = os.path.join(output_dir, "labels")
    if not os.path.exists(labels_dir):
        print("No labels directory found!")
        return False
    
    total_images = 0
    class_presence = defaultdict(int)
    
    for label_file in os.listdir(labels_dir):
        if not label_file.endswith('.txt'):
            continue
        
        label_path = os.path.join(labels_dir, label_file)
        classes = get_classes_from_label(label_path)
        
        total_images += 1
        for cls in classes:
            if cls in expected_classes:
                class_presence[cls] += 1
    
    print(f"\nVerified {total_images} images in output folder")
    print("\nClass presence in selected images:")
    for cls in sorted(expected_classes):
        count = class_presence.get(cls, 0)
        print(f"  Class {cls:2d}: {count:3d} images")
    
    return True

def main():
    """
    Main function to select images for each class.
    """
    print("="*60)
    print("IMAGE SELECTION TOOL")
    print("="*60)
    print(f"\nSource image directory: {IMAGE_DIR}")
    print(f"Source label directory: {LABEL_DIR}")
    print(f"Output directory: {OUTPUT_DIR}")
    print(f"Classes: {CLASSES}")
    print(f"Images per class: {IMAGES_PER_CLASS}")
    
    # Check if directories exist
    if not os.path.exists(IMAGE_DIR):
        print(f"\n❌ Error: Image directory not found: {IMAGE_DIR}")
        return
    
    if not os.path.exists(LABEL_DIR):
        print(f"\n❌ Error: Label directory not found: {LABEL_DIR}")
        return
    
    # Collect images by class
    class_to_images, image_classes = collect_images_by_class(IMAGE_DIR, LABEL_DIR, CLASSES)
    
    if not class_to_images:
        print("\n❌ No images found with labels!")
        return
    
    # Show available classes
    print(f"\nFound classes with images: {sorted(class_to_images.keys())}")
    
    # Select images per class
    selected_images, class_selection = select_images_per_class(class_to_images, IMAGES_PER_CLASS)
    
    if not selected_images:
        print("\n❌ No images selected!")
        return
    
    # Copy selected images
    copied_count, copied_classes = copy_selected_images(
        selected_images, IMAGE_DIR, LABEL_DIR, OUTPUT_DIR, image_classes
    )
    
    # Print statistics
    print_statistics(class_selection, copied_classes, OUTPUT_DIR)
    
    # Verify selection
    verify_selection(OUTPUT_DIR, CLASSES)
    
    print("\n" + "="*60)
    print("✅ PROCESS COMPLETED SUCCESSFULLY!")
    print("="*60)


# Update these paths
IMAGE_DIR = r"DIOR/dior_filtered_classes/train/images"  # <-- CHANGE THIS
LABEL_DIR = r"DIOR/dior_filtered_classes/train/labels"  # <-- CHANGE THIS
OUTPUT_DIR = r"DIOR/dior_filtered_classes/train_shot"  # <-- CHANGE THIS
CLASSES = [0, 4,7, 8]  # Specific classes to select
IMAGES_PER_CLASS = 7

main()
# Update these paths
IMAGE_DIR = r"DIOR/dior_filtered_classes/val/images"  # <-- CHANGE THIS
LABEL_DIR = r"DIOR/dior_filtered_classes/val/labels"  # <-- CHANGE THIS
OUTPUT_DIR = r"DIOR/dior_filtered_classes/val_shot"  # <-- CHANGE THIS
CLASSES = [0, 4,7, 8]  # Specific classes to select
IMAGES_PER_CLASS = 3

main()

IMAGE SELECTION TOOL

Source image directory: DIOR/dior_filtered_classes/train/images
Source label directory: DIOR/dior_filtered_classes/train/labels
Output directory: DIOR/dior_filtered_classes/train_shot
Classes: [0, 4, 7, 8]
Images per class: 7
Scanning images and labels...

Found classes with images: [0, 4, 7, 8]

SELECTING IMAGES FOR EACH CLASS

Class 0: 1377 images available
  Selecting 7 images

Class 4: 1318 images available
  Selecting 7 images

Class 7: 2178 images available
  Selecting 7 images

Class 8: 1135 images available
  Selecting 7 images

COPYING FILES

SELECTION STATISTICS

Per-class selection summary:
--------------------------------------------------
Class  0:  7/7 selected (Available: 1377) ✓
Class  4:  7/7 selected (Available: 1318) ✓
Class  7:  7/7 selected (Available: 2178) ✓
Class  8:  7/7 selected (Available: 1135) ✓

Total unique images selected: 4
Total image instances (counting multiple classes per image): 28

ACTUAL CLASS DISTRIBUTION IN COPIED IMAGES
(

In [4]:
import os
from pathlib import Path

for path in ['DIOR/dior_filtered_classes/train_shot/labels','DIOR/dior_filtered_classes/val_shot/labels','DIOR/dior_filtered_classes/test/labels',]:

    input_folder = Path(path)
    output_folder = input_folder.parent / (input_folder.name + '-new')

    # create output folder if it doesn't exist
    os.makedirs(output_folder, exist_ok=True)

    for filename in os.listdir(input_folder):
        if filename.endswith(".txt"):
            input_path = os.path.join(input_folder, filename)
            output_path = os.path.join(output_folder, filename)

            with open(input_path, "r") as f:
                lines = f.readlines()

            new_lines = []
            for line in lines:
                line = line.strip()
                if not line:
                    continue

                parts = line.split()
                parts[0] = "0"   # change class id to 0
                new_lines.append(" ".join(parts))

            with open(output_path, "w") as f:
                f.write("\n".join(new_lines))

print("All label files converted successfully.")

All label files converted successfully.


In [5]:
import os

# Configuration
DATASET_PATH = r"DIOR/dior_filtered_classes"  # Change this to your path
OUTPUT_FILE = os.path.join(DATASET_PATH, "data.yaml")
os.makedirs(DATASET_PATH, exist_ok=True)

# For 20 classes
data_yaml_content = f"""path: {DATASET_PATH}  
train: train_shot/images  
val: val_shot/images  
test: test/images  


nc: 1
names: [0]
"""

# Write to file
with open(OUTPUT_FILE, 'w') as f:
    f.write(data_yaml_content)

print(f"✅ data.yaml created at: {OUTPUT_FILE}")
print("\nContent:")
print(data_yaml_content)

✅ data.yaml created at: DIOR/dior_filtered_classes/data.yaml

Content:
path: DIOR/dior_filtered_classes  
train: train_shot/images  
val: val_shot/images  
test: test/images  


nc: 1
names: [0]



In [6]:
import os
import cv2
import numpy as np
from pathlib import Path

# Configuration
IMAGE_DIR = r"/home/gpu_03/class-agnostic/DIOR/dior_filtered_classes/train_shot/images"
LABEL_DIR = r"/home/gpu_03/class-agnostic/DIOR/dior_filtered_classes/train_shot/labels"
OUTPUT_DIR = r"/home/gpu_03/class-agnostic/DIOR/dior_filtered_classes/train_shot/images_with_boxes"
# CLASS_NAMES = [f"class_{i}" for i in range(20)]  # Replace with actual class names if needed
CLASS_NAMES = ['7', '8', '13', '17']  # Specific classes to select

# Colors for each class (BGR format)
np.random.seed(42)
COLORS = [tuple(np.random.randint(0, 255, 3).tolist()) for _ in range(20)]

def draw_yolo_boxes(image_path, label_path, output_path, class_names, colors):
    """
    Draw bounding boxes from YOLO label file onto image and save.
    """
    # Load image
    image = cv2.imread(image_path)
    if image is None:
        print(f"Error: Could not read {image_path}")
        return False
    
    height, width = image.shape[:2]
    
    # Read label file
    if not os.path.exists(label_path):
        print(f"Warning: Label file {label_path} not found")
        cv2.imwrite(output_path, image)
        return True
    
    with open(label_path, 'r') as f:
        lines = f.readlines()
    
    # Process each bounding box
    for line in lines:
        line = line.strip()
        if not line:
            continue
        
        parts = line.split()
        if len(parts) != 5:
            print(f"Warning: Invalid format in {label_path}: {line}")
            continue
        
        class_id = int(parts[0])
        center_x = float(parts[1]) * width
        center_y = float(parts[2]) * height
        box_width = float(parts[3]) * width
        box_height = float(parts[4]) * height
        
        # Convert YOLO format to pixel coordinates
        x1 = int(center_x - box_width / 2)
        y1 = int(center_y - box_height / 2)
        x2 = int(center_x + box_width / 2)
        y2 = int(center_y + box_height / 2)
        
        # Ensure coordinates are within image bounds
        x1 = max(0, x1)
        y1 = max(0, y1)
        x2 = min(width, x2)
        y2 = min(height, y2)
        
        # Draw rectangle
        color = colors[class_id % len(colors)]
        cv2.rectangle(image, (x1, y1), (x2, y2), color, 2)
        
        # Add label text
        label = class_names[class_id] if class_id < len(class_names) else f"class_{class_id}"
        font = cv2.FONT_HERSHEY_SIMPLEX
        font_scale = 0.5
        thickness = 1
        text_size = cv2.getTextSize(label, font, font_scale, thickness)[0]
        
        # Draw background rectangle for text
        cv2.rectangle(image, (x1, y1 - text_size[1] - 4), (x1 + text_size[0] + 4, y1), color, -1)
        
        # Draw text
        cv2.putText(image, label, (x1 + 2, y1 - 2), font, font_scale, (255, 255, 255), thickness)
    
    # Save output image
    cv2.imwrite(output_path, image)
    return True

def process_all_images(image_dir, label_dir, output_dir, class_names, colors):
    """
    Process all image-label pairs in the directories.
    """
    # Create output directory
    os.makedirs(output_dir, exist_ok=True)
    
    # Get all image files
    image_extensions = {'.jpg', '.jpeg', '.png', '.bmp', '.tiff'}
    image_files = [f for f in os.listdir(image_dir) 
                   if Path(f).suffix.lower() in image_extensions]
    
    success_count = 0
    for img_file in sorted(image_files):
        # Get base name without extension
        base_name = Path(img_file).stem
        img_path = os.path.join(image_dir, img_file)
        
        # Find corresponding label file
        label_file = f"{base_name}.txt"
        label_path = os.path.join(label_dir, label_file)
        
        # Output path
        output_path = os.path.join(output_dir, img_file)
        
        # Process
        if draw_yolo_boxes(img_path, label_path, output_path, class_names, colors):
            success_count += 1
            print(f"Processed: {img_file}")
    
    print(f"\nCompleted: {success_count}/{len(image_files)} images processed")
    print(f"Output saved to: {output_dir}")

# Run the script
if __name__ == "__main__":
    process_all_images(IMAGE_DIR, LABEL_DIR, OUTPUT_DIR, CLASS_NAMES, COLORS)

Processed: 02193.jpg
Processed: 02261.jpg
Processed: 04630.jpg
Processed: 05229.jpg
Processed: 05637.jpg
Processed: 07904.jpg
Processed: 08134.jpg
Processed: 08701.jpg
Processed: 08834.jpg
Processed: 10804.jpg
Processed: 11263.jpg
Processed: 12119.jpg
Processed: 12855.jpg
Processed: 14078.jpg
Processed: 15933.jpg
Processed: 15971.jpg
Processed: 16049.jpg
Processed: 16450.jpg
Processed: 16890.jpg
Processed: 17805.jpg
Processed: 18904.jpg
Processed: 19441.jpg
Processed: 20072.jpg
Processed: 20656.jpg
Processed: 20680.jpg
Processed: 20731.jpg
Processed: 21181.jpg
Processed: 21386.jpg

Completed: 28/28 images processed
Output saved to: /home/gpu_03/class-agnostic/DIOR/dior_filtered_classes/train_shot/images_with_boxes


In [ ]:
import cv2
import os

# Paths
img_path = '/home/gpu_03/class-agnostic/DIOR/dior_filtered_classes/train/images/08701.jpg'
label_path = '/home/gpu_03/class-agnostic/DIOR/dior_filtered_classes/train/labels/08701.txt'
save_dir = '/home/gpu_03/class-agnostic/DIOR/dior_filtered_classes/train_box'

# Create save directory if it doesn't exist
os.makedirs(save_dir, exist_ok=True)

# Class names for DIOR dataset (20 classes)
class_names = ['']

def convert_yolo_to_bbox(yolo_coords, img_width, img_height):
    """Convert YOLO format (cx, cy, w, h) normalized to absolute coordinates"""
    cx, cy, w, h = yolo_coords
    x1 = int((cx - w/2) * img_width)
    y1 = int((cy - h/2) * img_height)
    x2 = int((cx + w/2) * img_width)
    y2 = int((cy + h/2) * img_height)
    # Clamp to image boundaries
    x1 = max(0, min(x1, img_width))
    x2 = max(0, min(x2, img_width))
    y1 = max(0, min(y1, img_height))
    y2 = max(0, min(y2, img_height))
    return x1, y1, x2, y2

# Load image
image = cv2.imread(img_path)
if image is None:
    print(f"Error: Could not read image from {img_path}")
    exit()

img_height, img_width = image.shape[:2]

# Load ground truth boxes from label file
gt_boxes = []
if os.path.exists(label_path):
    with open(label_path, 'r') as f:
        lines = f.readlines()
        print(f"Found {len(lines)} objects in label file")
        
        for line in lines:
            parts = line.strip().split()
            if len(parts) >= 5:
                class_id = int(parts[0])
                yolo_coords = [float(parts[1]), float(parts[2]), float(parts[3]), float(parts[4])]
                x1, y1, x2, y2 = convert_yolo_to_bbox(yolo_coords, img_width, img_height)
                gt_boxes.append([x1, y1, x2, y2, class_id])
                print(f"  Box {len(gt_boxes)}: class={class_names[class_id]}, coords=[{x1},{y1},{x2},{y2}]")
else:
    print(f"Warning: Label file not found at {label_path}")

# Draw ground truth boxes on image
for box in gt_boxes:
    x1, y1, x2, y2, class_id = box
    class_name = class_names[class_id]
    
    # Draw rectangle (green color, thickness 2)
    cv2.rectangle(image, (x1, y1), (x2, y2), (0, 255, 0), 2)
    
    # Prepare label text
    label = f"{class_name}"
    
    # Calculate text size for background
    (label_w, label_h), baseline = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.6, 2)
    
    # Draw background rectangle for text
    cv2.rectangle(image, (x1, y1 - label_h - 5), (x1 + label_w, y1), (0, 255, 0), -1)
    
    # Draw text
    cv2.putText(image, label, (x1, y1 - 3), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 0), 2)

# Add info text at the top
info_text = f"Total Objects: {len(gt_boxes)}"
cv2.putText(image, info_text, (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)

# Save the image with boxes
save_path = os.path.join(save_dir, os.path.basename(img_path))
cv2.imwrite(save_path, image)

print(f"\n✓ Image saved to: {save_path}")
print(f"✓ Total {len(gt_boxes)} ground truth boxes drawn")

Found 53 objects in label file
  Box 1: class=dam, coords=[1,254,391,761]
  Box 2: class=dam, coords=[161,142,593,620]
  Box 3: class=dam, coords=[286,0,799,604]
  Box 4: class=Expressway-Service-area, coords=[327,659,390,723]
  Box 5: class=Expressway-Service-area, coords=[265,596,297,627]
  Box 6: class=Expressway-Service-area, coords=[211,527,267,584]
  Box 7: class=Expressway-Service-area, coords=[175,467,210,497]
  Box 8: class=Expressway-Service-area, coords=[134,388,191,442]
  Box 9: class=Expressway-Service-area, coords=[150,429,181,452]
  Box 10: class=Expressway-Service-area, coords=[160,450,192,475]
  Box 11: class=Expressway-Service-area, coords=[123,405,139,424]
  Box 12: class=Expressway-Service-area, coords=[72,342,100,368]
  Box 13: class=Expressway-Service-area, coords=[40,306,70,333]
  Box 14: class=Expressway-Service-area, coords=[24,254,93,315]
  Box 15: class=Expressway-Service-area, coords=[481,457,529,505]
  Box 16: class=Expressway-Service-area, coords=[563,545,